# Asset Pricing Tests for Size and Momentum Portfolios

In this notebook, we will replicate the relevant parts of Tables 6 and 7 from Fama and French (2012).

This analysis is equivalent to the analysis in the previous notebook. The main difference is the portfolio dataset.

The previous notebook used portfolios formed using:

- Size
- Book-to-market

This notebook uses portfolios formed using:

- Size
- Momentum

Table 6 reports overall model performance across the portfolios.

Table 7 reports the alpha and alpha t-statistic for each individual portfolio.

The paper reports only the four-factor model.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

In [2]:
DATA_DIR = Path("cleaned_data")

## Load the factor data

The factor data are the same as those used in the previous notebook.

The three-factor files contain:

- `Mkt-RF`
- `SMB`
- `HML`
- `RF`

The momentum files contain:

- `WML`

In [3]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [4]:
developed_momentum = pd.read_csv(
    DATA_DIR / "developed_momentum.csv",
    parse_dates=["date"]
)

In [5]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [6]:
japan_momentum = pd.read_csv(
    DATA_DIR / "japan_momentum.csv",
    parse_dates=["date"]
)

## Load the size and momentum portfolios

The portfolio files contain value-weighted monthly returns for 25 portfolios.

The portfolios are formed using:

- Five company-size groups
- Five past-return groups

This gives 5 × 5 = 25 portfolios.

The past-return groups range from past losers to past winners.

In [7]:
developed_portfolios = pd.read_csv(
    DATA_DIR / "developed_25_size_momentum.csv",
    parse_dates=["date"]
)

In [8]:
japan_portfolios = pd.read_csv(
    DATA_DIR / "japan_25_size_momentum.csv",
    parse_dates=["date"]
)

## Add the momentum factor

Merge the three-factor and momentum files using `date`.

The `one_to_one` check confirms that each month appears only once in each dataset.

In [9]:
developed_factors = developed_factors.merge(
    developed_momentum,
    on="date",
    validate="one_to_one"
)

In [10]:
japan_factors = japan_factors.merge(
    japan_momentum,
    on="date",
    validate="one_to_one"
)

## Check the datasets

The sample runs from November 1990 to March 2011.

Each dataset should contain 245 monthly observations.

The factor datasets should contain six columns.

The portfolio datasets should contain one date column and 25 portfolio return columns.

In [11]:
print("Developed factors:", developed_factors.shape)
print("Japanese factors:", japan_factors.shape)
print("Developed portfolios:", developed_portfolios.shape)
print("Japanese portfolios:", japan_portfolios.shape)

Developed factors: (245, 6)
Japanese factors: (245, 6)
Developed portfolios: (245, 26)
Japanese portfolios: (245, 26)


## Select the portfolios

The 5x5 results use all 25 portfolios.

The 4x5 results remove the five portfolios in the smallest size group.

The first five portfolio columns belong to the smallest size group.

In [12]:
all_portfolios = developed_portfolios.columns.drop("date").tolist()

In [13]:
without_microcaps = all_portfolios[5:]

In [14]:
print("Number of 5x5 portfolios:", len(all_portfolios))
print("Number of 4x5 portfolios:", len(without_microcaps))

Number of 5x5 portfolios: 25
Number of 4x5 portfolios: 20


## Define the model

Tables 6 and 7 report only the four-factor model.

The four-factor model uses:

- `Mkt-RF`
- `SMB`
- `HML`
- `WML`

In [15]:
models = {
    "Four-factor": ["Mkt-RF", "SMB", "HML", "WML"]
}

## Table 6

Table 6 is equivalent to Table 3 in the previous notebook.

The dependent variables are now the excess returns of the size and momentum portfolios.

For each portfolio:

1. Subtract `RF` from the portfolio return.
2. Run the four-factor regression.
3. Store the fitted regression result.

For the 5x5 cases, Table 6 reports:

- `GRS`
- `|a|`
- `Adjusted R2`
- `s(a)`
- `SR(a)`

These results use all 25 portfolios.

For the 4x5 cases, Table 6 reports only:

- `GRS`
- `|a|`
- `SR(a)`

These results use the 20 portfolios remaining after the smallest size group is removed.

The paper does not report `Adjusted R2` or `s(a)` for the 4x5 cases.

## Task 1: Global portfolios with Global factors, 5x5

Use:

- `developed_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

Run a separate four-factor regression for every portfolio.

Store the regression outputs and calculate the five Table 6 statistics.

Print the result for the four-factor model.

Compare your values with the Global 5x5 part of Table 6. State whether the values match and briefly interpret the result.

In [16]:
# Shared helpers, used by every task in this notebook (same as in notebook 03,
# repeated here so this notebook runs on its own).

T_EXPECTED = 245

FIVE_BY_FIVE = ["GRS", "|a|", "Adjusted R2", "s(a)", "SR(a)"]
FOUR_BY_FIVE = ["GRS", "|a|", "SR(a)"]


def prepare_data(portfolios, factors, portfolio_columns):
    """Merge on date and return the excess portfolio returns and the matched factor data."""
    overlap = portfolios["date"].isin(factors["date"]).sum()
    print("Overlapping dates between portfolios and factors:", overlap)
    if overlap != T_EXPECTED:
        raise ValueError(f"Expected {T_EXPECTED} overlapping dates, found {overlap}. Stop.")

    merged = portfolios.merge(factors, on="date", validate="one_to_one")
    excess = merged[portfolio_columns].sub(merged["RF"], axis=0)
    return excess, merged


def run_model(excess, merged, factor_columns):
    """One OLS regression per portfolio: excess return on a constant plus the model's factors."""
    X = sm.add_constant(merged[factor_columns])
    return {portfolio: sm.OLS(excess[portfolio], X).fit() for portfolio in excess.columns}


def check_results(results, n_portfolios):
    """Mechanical checks: right number of regressions, 245 observations, adjusted R2 in [0, 1]."""
    if len(results) != n_portfolios:
        raise ValueError(f"Expected {n_portfolios} regressions, found {len(results)}")
    for portfolio, result in results.items():
        if result.nobs != T_EXPECTED:
            raise ValueError(f"{portfolio}: {result.nobs} observations, expected {T_EXPECTED}")
        if not 0 <= result.rsquared_adj <= 1:
            raise ValueError(f"{portfolio}: adjusted R2 {result.rsquared_adj:.3f} outside [0, 1]")


def alphas_and_residuals(results):
    """Stack the N alphas into a vector and the residuals into a T x N matrix."""
    alphas = np.array([result.params["const"] for result in results.values()])
    residuals = np.column_stack([result.resid for result in results.values()])
    return alphas, residuals


def residual_covariance(residuals):
    """N x N covariance matrix of the regression residuals (divided by T)."""
    T = residuals.shape[0]
    return residuals.T @ residuals / T


def grs_statistic(results, factor_returns):
    """Gibbons, Ross and Shanken (1989) test that all N alphas are jointly zero.

    GRS = (T - N - K) / N * (a' S^-1 a) / (1 + f' W^-1 f)
    where S is the residual covariance matrix, f the vector of factor means and
    W the factor covariance matrix (both covariances divided by T).
    """
    alphas, residuals = alphas_and_residuals(results)
    T, N = residuals.shape
    K = factor_returns.shape[1]

    S = residual_covariance(residuals)
    f_mean = factor_returns.mean().to_numpy()
    f_demeaned = factor_returns.to_numpy() - f_mean
    W = f_demeaned.T @ f_demeaned / T

    alpha_term = alphas @ np.linalg.solve(S, alphas)
    factor_term = f_mean @ np.linalg.solve(W, f_mean)
    return (T - N - K) / N * alpha_term / (1 + factor_term)


def sr_alpha(results):
    """Sharpe ratio of the intercepts, equation (3) in the paper: SR(a) = (a' S^-1 a)^(1/2)."""
    alphas, residuals = alphas_and_residuals(results)
    S = residual_covariance(residuals)
    return np.sqrt(alphas @ np.linalg.solve(S, alphas))


def model_statistics(results, factor_returns):
    """Table 6 statistics for one model on one set of portfolios."""
    alphas, _ = alphas_and_residuals(results)
    return {
        "GRS": grs_statistic(results, factor_returns),
        "|a|": np.mean(np.abs(alphas)),
        "Adjusted R2": np.mean([result.rsquared_adj for result in results.values()]),
        "s(a)": np.mean([result.bse["const"] for result in results.values()]),
        "SR(a)": sr_alpha(results),
    }


def scorecard(portfolios, factors, portfolio_columns, models, statistics):
    """Run every model on the selected portfolios.

    Returns the fitted regressions (model name -> portfolio -> result) and one
    table with a row per model and the requested statistics as columns.
    """
    excess, merged = prepare_data(portfolios, factors, portfolio_columns)

    fitted = {}
    rows = {}
    for model_name, factor_columns in models.items():
        results = run_model(excess, merged, factor_columns)
        check_results(results, len(portfolio_columns))
        fitted[model_name] = results
        rows[model_name] = model_statistics(results, merged[factor_columns])

    print(f"Checks passed: {len(portfolio_columns)} regressions per model, "
          f"{T_EXPECTED} observations each, adjusted R2 between 0 and 1")
    table = pd.DataFrame(rows).T[statistics]
    return fitted, table.round(2)


# Task 1: Global portfolios, Global factors, 5x5

global_fitted, global_5x5 = scorecard(
    developed_portfolios, developed_factors, all_portfolios, models, FIVE_BY_FIVE
)
print("Global size-momentum portfolios with Global factors, 5x5")
global_5x5

Overlapping dates between portfolios and factors: 245
Checks passed: 25 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Global size-momentum portfolios with Global factors, 5x5


,GRS,|a|,Adjusted R2,s(a),SR(a)
Four-factor,3.86,0.14,0.94,0.08,0.71


## Task 2: Global portfolios with Global factors, 4x5

Use:

- `developed_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

Run a separate four-factor regression for every portfolio.

Store the regression outputs.

Report the three statistics shown in the 4x5 part of Table 6:

- `GRS`
- `|a|`
- `SR(a)`

Compare your values with the Global 4x5 part of Table 6. Discuss whether removing the smallest portfolios improves the model.

In [17]:
global_4x5_fitted, global_4x5 = scorecard(
    developed_portfolios, developed_factors, without_microcaps, models, FOUR_BY_FIVE
)
print("Global size-momentum portfolios with Global factors, 4x5 (without microcaps)")
global_4x5

Overlapping dates between portfolios and factors: 245
Checks passed: 20 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Global size-momentum portfolios with Global factors, 4x5 (without microcaps)


,GRS,|a|,SR(a)
Four-factor,2.03,0.09,0.45


## Task 3: Japanese portfolios with Global factors, 5x5

Use:

- `japan_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

Run a separate four-factor regression for every Japanese portfolio using Global Developed factors.

Store the regression outputs and calculate the five Table 6 statistics.

Print the result for the four-factor model.

Compare your values with the Japan, Global factors, 5x5 part of Table 6. Interpret how well Global factors explain Japanese size and momentum portfolio returns.

In [18]:
japan_global_fitted, japan_global_5x5 = scorecard(
    japan_portfolios, developed_factors, all_portfolios, models, FIVE_BY_FIVE
)
print("Japanese size-momentum portfolios with Global factors, 5x5")
japan_global_5x5

Overlapping dates between portfolios and factors: 245


Checks passed: 25 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese size-momentum portfolios with Global factors, 5x5


,GRS,|a|,Adjusted R2,s(a),SR(a)
Four-factor,1.64,0.62,0.35,0.38,0.46


## Task 4: Japanese portfolios with Global factors, 4x5

Use:

- `japan_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

Run a separate four-factor regression for every Japanese portfolio using Global Developed factors.

Store the regression outputs.

Report the three statistics shown in the 4x5 part of Table 6:

- `GRS`
- `|a|`
- `SR(a)`

Compare your values with the Japan, Global factors, 4x5 part of Table 6. Discuss whether removing the smallest Japanese portfolios changes the result.

In [19]:
japan_global_4x5_fitted, japan_global_4x5 = scorecard(
    japan_portfolios, developed_factors, without_microcaps, models, FOUR_BY_FIVE
)
print("Japanese size-momentum portfolios with Global factors, 4x5 (without microcaps)")
japan_global_4x5

Overlapping dates between portfolios and factors: 245


Checks passed: 20 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese size-momentum portfolios with Global factors, 4x5 (without microcaps)


,GRS,|a|,SR(a)
Four-factor,1.32,0.67,0.37


## Task 5: Japanese portfolios with Japanese factors, 5x5

Use:

- `japan_portfolios`
- `japan_factors`
- All 25 portfolios in `all_portfolios`

Run a separate four-factor regression for every Japanese portfolio using Japanese factors.

Store the regression outputs and calculate the five Table 6 statistics.

Print the result for the four-factor model.

Compare your values with the Japan, Local factors, 5x5 part of Table 6. Compare this result with Task 3.

In [20]:
japan_local_fitted, japan_local_5x5 = scorecard(
    japan_portfolios, japan_factors, all_portfolios, models, FIVE_BY_FIVE
)
print("Japanese size-momentum portfolios with Japanese factors, 5x5")
japan_local_5x5

Overlapping dates between portfolios and factors: 245
Checks passed: 25 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese size-momentum portfolios with Japanese factors, 5x5


,GRS,|a|,Adjusted R2,s(a),SR(a)
Four-factor,1.04,0.11,0.93,0.12,0.35


## Task 6: Japanese portfolios with Japanese factors, 4x5

Use:

- `japan_portfolios`
- `japan_factors`
- The 20 portfolios in `without_microcaps`

Run a separate four-factor regression for every Japanese portfolio using Japanese factors.

Store the regression outputs.

Report the three statistics shown in the 4x5 part of Table 6:

- `GRS`
- `|a|`
- `SR(a)`

Compare your values with the Japan, Local factors, 4x5 part of Table 6.

Discuss:

1. Whether removing the smallest portfolios changes the results.
2. Whether Global or Japanese factors explain Japanese portfolio returns better.

In [21]:
japan_local_4x5_fitted, japan_local_4x5 = scorecard(
    japan_portfolios, japan_factors, without_microcaps, models, FOUR_BY_FIVE
)
print("Japanese size-momentum portfolios with Japanese factors, 4x5 (without microcaps)")
japan_local_4x5

Overlapping dates between portfolios and factors: 245


Checks passed: 20 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese size-momentum portfolios with Japanese factors, 4x5 (without microcaps)


,GRS,|a|,SR(a)
Four-factor,0.83,0.09,0.28


# Table 7: Individual Portfolio Alphas

Table 7 is equivalent to Table 4 in the previous notebook.

Table 6 summarizes the results across all 25 or 20 portfolio regressions.

Table 7 reports the alpha and alpha t-statistic for each individual portfolio.

For each portfolio, report:

- `a`: The regression alpha
- `t(a)`: The t-statistic of the alpha

These values are taken directly from the stored regression outputs.

The values are not averaged across portfolios.

## Task 7: Global portfolio alphas using Global factors

You have already estimated and stored the regression outputs for Global portfolios using Global factors in Task 1.

Use the stored 5x5 four-factor regression results.

Report:

- The alpha of each portfolio
- The t-statistic of each alpha

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with:

`Global size-momentum returns regressed on global factors`

in Table 7.

Identify any patterns across size and past-return groups.

In [22]:
from IPython.display import display

SIZE_LABELS = ["Small", "2", "3", "4", "Big"]
MOMENTUM_LABELS = ["Losers", "2", "3", "4", "Winners"]


def alpha_grids(results, column_labels):
    """The 25 alphas and their t-statistics, each arranged 5x5 with rows Small to Big."""
    if len(results) != 25:
        raise ValueError(f"Expected 25 regressions, found {len(results)}")
    alphas = [result.params["const"] for result in results.values()]
    t_stats = [result.tvalues["const"] for result in results.values()]

    alpha_grid = pd.DataFrame(np.reshape(alphas, (5, 5)), index=SIZE_LABELS, columns=column_labels)
    t_grid = pd.DataFrame(np.reshape(t_stats, (5, 5)), index=SIZE_LABELS, columns=column_labels)
    return alpha_grid.round(2), t_grid.round(2)


alpha_grid, t_grid = alpha_grids(global_fitted["Four-factor"], MOMENTUM_LABELS)
global_alpha_grids = {"Four-factor": {"a": alpha_grid, "t(a)": t_grid}}

print("Global size-momentum returns regressed on global factors: Four-factor")
display(pd.concat({"a": alpha_grid, "t(a)": t_grid}, axis=1))

Global size-momentum returns regressed on global factors: Four-factor


a                             t(a)                          
      Losers     2     3     4 Winners Losers     2     3     4 Winners
Small  -0.07  0.10  0.19  0.46    0.76  -0.64  1.32  2.52  6.02    6.40
2      -0.05 -0.04 -0.08  0.12    0.29  -0.63 -0.68 -1.07  1.78    3.77
3       0.09 -0.06 -0.08 -0.15    0.00   1.04 -0.78 -1.14 -2.11    0.00
4       0.13 -0.04 -0.04 -0.15    0.02   1.28 -0.62 -0.63 -2.16    0.23
Big     0.19  0.02 -0.09 -0.11   -0.15   1.94  0.32 -1.31 -1.93   -1.65

## Task 8: Japanese portfolio alphas using Japanese factors

You have already estimated and stored the regression outputs for Japanese portfolios using Japanese factors in Task 5.

Use the stored 5x5 four-factor regression results.

Report:

- The alpha of each portfolio
- The t-statistic of each alpha

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with:

`Japanese size-momentum returns regressed on Japanese factors`

in Table 7.

In [23]:
alpha_grid, t_grid = alpha_grids(japan_local_fitted["Four-factor"], MOMENTUM_LABELS)
japan_alpha_grids = {"Four-factor": {"a": alpha_grid, "t(a)": t_grid}}

print("Japanese size-momentum returns regressed on Japanese factors: Four-factor")
display(pd.concat({"a": alpha_grid, "t(a)": t_grid}, axis=1))

Japanese size-momentum returns regressed on Japanese factors: Four-factor


a                             t(a)                          
      Losers     2     3     4 Winners Losers     2     3     4 Winners
Small   0.31  0.32  0.12  0.24    0.04   2.14  2.54  1.16  2.05    0.21
2       0.04 -0.02 -0.02 -0.06   -0.03   0.34 -0.22 -0.21 -0.51   -0.20
3      -0.06 -0.21 -0.14 -0.01    0.03  -0.51 -2.09 -1.19 -0.11    0.21
4       0.04 -0.08 -0.14 -0.15    0.05   0.30 -0.69 -1.14 -1.23    0.33
Big     0.10 -0.19 -0.25 -0.11    0.01   0.63 -1.59 -1.97 -1.06    0.12